
## Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

• Tracking agent behavior with logging, analytics, and debugging.

• Transforming prompts, tool selection, and output formatting.

• Adding retries, fallbacks, and early termination logic.

• Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# Loading Groq API 
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

• Long-running conversations that exceed context windows.

• Multi-turn dialogues with extensive history.

• Applications where preserving full conversation context matters.

In [6]:
from langchain.agents import create_agent 
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

model_1=ChatGroq(model="llama-3.1-8b-instant")
model_2=ChatGroq(model="qwen/qwen3-32b")


agent=create_agent(
    model=model_2,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model_1,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [5]:
## Runn with a thread ID
config = {"configurable": {"thread_id": "abc-123"}}

In [7]:
# Alternate test data

queries = [
    "What is 2+2?",
    "What is 100/4?",
    "What is 0.25*3?",
    "What is 2**5?",
    "What is 15+8-3?",
    "What is (7-2)/5?",
    "What is 12+32?",
    "What is 15-6+2?"
]

for ques in queries:
    response = agent.invoke({"messages": [HumanMessage(content=ques)]}, config=config)
    print(f"Q: {ques}\nA: {response}\n length of messages in memory: {len(response['messages'])}\n{'-'*50}")

Q: What is 2+2?
A: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='140115dd-58a2-4e23-b525-bfd7a7130fcd'), AIMessage(content='<think>\nOkay, so the user is asking "What is 2+2?" Hmm, that\'s a basic arithmetic question. Let me think. I remember from school that 2 plus 2 is 4. But maybe I should make sure I\'m not missing anything here. Is there any trick or context I should consider? Sometimes people ask this as a joke or to test if the responder is paying attention. But in standard mathematics, addition is straightforward. 2 is a whole number, and adding two of them together would just give 4. Let me visualize it: if I have two apples and someone gives me two more apples, how many do I have? Four apples. Yeah, that checks out. I don\'t think there\'s any ambiguity here unless the question is in a different numeral system, like binary or something. Wait, if it\'s binary, then 2 in binary is actually 10, so 10 + 10 would be 100, which i

### Middleware: Token Size

In [10]:
from langchain.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search Hotels: return long response to use mode tokens."""

    return f"""Hotels in {city}:
        1. Grand Hotel: 5-stars, $350/night, spa, pool, gym
        2. City Inn: 4-stars, $200/night, free breakfast, business center
        3. Budget Stay: 3-stars, $100/night, free Wi-Fi, parking"""


agent=create_agent(
    model=model_2,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model_1,
            trigger=("tokens",500),
            keep=("tokens",200)
        )
    ]
)

## Runn with a thread ID
config1 = {"configurable": {"thread_id": "abc-124"}}

# Token Counter Approximate
def count_tokens(messages):
    total_chars=sum(len(msg.content) for msg in messages)
    return total_chars // 4  # Approximate token count (assuming ~4 chars per token)

In [11]:
cities= ["Paris", "London", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find the hotels in {city}")]},
        config=config1
    )

    tokens = count_tokens(response["messages"])

    print(f"{city}: ~{tokens} Tokens in memory. {len(response['messages'])} messages.")
    print(f"Response: {response['messages']}\n{'-'*50}")

Paris: ~163 Tokens in memory. 4 messages.
Response: [HumanMessage(content='find the hotels in Paris', additional_kwargs={}, response_metadata={}, id='00ae5fa1-9824-4ecc-9297-d8f91b838271'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking to find hotels in Paris. Let me check the tools provided. There\'s a function called search_hotels that takes a city parameter. The required parameter is city, and it\'s a string. Since the user mentioned Paris, I need to call this function with "Paris" as the city argument. I\'ll make sure to format the tool call correctly within the XML tags as specified. No other functions are available, so this should be the only tool call needed. Let me double-check the syntax for the JSON object to ensure it\'s correct.\n', 'tool_calls': [{'id': 'h0hwgc67n', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_toke

### Fraction

In [14]:
@tool
def hotels_search(city: str)-> str:
    """Search Hotels."""
    return f"Hotels in {city}: Grand Hotels $350/night, City Inn $200/night, Budget Stays $75/night"

agent=create_agent(
    model=model_2,
    tools=[hotels_search],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model_1,
            trigger=("fraction",0.005), # 0.5% = ~640 tokens for 128k context
            keep=("fraction",0.002)     # 0.2% = ~256 tokens for 128k context
        )
    ]
)

## Runn with a thread ID
config2 = {"configurable": {"thread_id": "abc-125"}}


# Token Counter Approximate
def token_count(messages):
    return sum(len(str(m.content)) for m in messages) // 4  # Approximate token count (assuming ~4 chars per token)

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config2
    )
    tokens = token_count(response["messages"])
    fraction = tokens / 128000
    print(f"{city}: ~{tokens} Tokens ({fraction:.4%}) in memory, {len(response['messages'])} messages.")
    print(f"Response: {response['messages']}\n{'-'*50}")

Paris: ~73 Tokens (0.0570%) in memory, 4 messages.
Response: [HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='d481b66f-8340-4bbb-968b-1d1a0dc7919b'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for hotels in Paris. Let me check the available tools. There's a function called hotels_search that takes a city parameter. The required parameter is city, and it's a string. Since the user mentioned Paris, I need to call this function with the city set to Paris. I'll make sure the JSON is correctly formatted with the city name in quotes. No other parameters are needed. Alright, time to structure the tool call.\n", 'tool_calls': [{'id': 'etvgyk2jc', 'function': {'arguments': '{"city":"Paris"}', 'name': 'hotels_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 147, 'total_tokens': 265, 'completion_time': 0.194069491, 'completion_tokens_details': {'reason

### Human in the Middleware Loop

Pause agents execution for an human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

* High-stakes operations requiring human approval (e.g. database writes, Financial transactions)
* Compliance workflows where human oversight is mandatory.
* Long-running conversations where human feedback guides the agent.

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [25]:
## Defining the function to be used as a tool

def read_email_tool(email_id: str) ->str:
    """Mock Function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock Function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}' and body '{body}'"


agent=create_agent(
    model=model_2,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {"allowed_decisions": ["approve","edit","reject"]},
                "read_email_tool": False,
            }
        )
    ]
)

#### Approve

In [26]:
config4 = {"configurable": {"thread_id": "abc-1565"}}

## Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@example.com with subject 'Meeting Reminder' and body 'Please remember our meeting tomorrow at 10 AM.'")]},
    config=config4
)

In [27]:
result

{'messages': [HumanMessage(content="Send email to john@example.com with subject 'Meeting Reminder' and body 'Please remember our meeting tomorrow at 10 AM.'", additional_kwargs={}, response_metadata={}, id='57e02aa1-c0c4-434a-9dab-8eed708b446f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@example.com. Let me check the available tools. There's the send_email_tool. The parameters required are recipient, subject, and body. The user provided all three: recipient is john@example.com, subject is 'Meeting Reminder', and the body is 'Please remember our meeting tomorrow at 10 AM.' So I need to call the send_email_tool with these arguments. No need to use the read_email_tool here since the request is about sending, not reading. Just structure the JSON with the parameters and make sure all required fields are included. That should do it.\n", 'tool_calls': [{'id': 'vx397zj8w', 'function': {'arguments': '{"body":"Please remember o

In [28]:
# Step 2: Human in the loop decision: Approve

from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! For human decision: approve, edit, reject")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[{"type":"approve"}]
            }
        ),
        config=config4
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! For human decision: approve, edit, reject
Result: The email has been successfully sent to john@example.com with the subject "Meeting Reminder" and the specified message. Let me know if you need anything else!


#### Reject 

In [29]:
config5 = {"configurable": {"thread_id": "abc-1555"}}

## Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@example.com with subject 'Meeting Reminder' and body 'Please remember our meeting tomorrow at 10 AM.'")]},
    config=config5
)

if "__interrupt__" in result:
    print("Paused! For human decision: approve, edit, reject")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[{"type":"reject"}]
            }
        ),
        config=config5
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! For human decision: approve, edit, reject
Result: It seems there was an issue with sending the email through the tool. Since this is a mock function, the rejection might be part of a simulated scenario. Would you like me to:

1. Attempt to resend the email with the same details?
2. Modify any part of the email (subject/body) and try again?
3. Check if there's a different "email ID" to use for this action?

Let me know how you'd like to proceed!


#### Editing

In [31]:
config6 = {"configurable": {"thread_id": "abc-1559"}}

## Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@example.com with subject 'Meeting Reminder' and body 'Please remember our meeting tomorrow at 10 AM.'")]},
    config=config6
)

if "__interrupt__" in result:
    print("Paused! For human decision: approve, edit, reject")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[{
                    "type":"edit",
                    "edited_action":{
                        "name": "send_email_tool",
                        "args": {
                            "recipient": "correct_email@example.com",
                            "subject": "Updated Meeting Reminder",
                            "body": "(Edited) Please remember our meeting tomorrow at 11 AM."
                        }
                    }
                }]
            }
        ),
        config=config6
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! For human decision: approve, edit, reject
Result: The email has been successfully sent to correct_email@example.com with the updated subject "Updated Meeting Reminder" and edited body text. Let me know if you'd like me to review the changes or take any other action!
